# 00 . Preparación de ubicaciones 

##  1. Carga de datos

In [3]:
import pandas as pd
from pathlib import Path
import re
#Importar las librerías necesarias.

In [4]:
DATA_DIR = Path("../data/raw/ubicacion-trafico")

dfs = []

for file in DATA_DIR.glob("*.csv"):
    df = pd.read_csv(file, encoding="cp1252", sep=";")
    df["archivo"] = file.name
    dfs.append(df)

ubicaciones = pd.concat(dfs, ignore_index=True)
#Cargar y consolidar los archivos CSV mensuales.

## 2. Exploración inicial

In [5]:
ubicaciones.info()
ubicaciones.head()
#Realizar una inspección inicial de los datos consolidados.

<class 'pandas.DataFrame'>
RangeIndex: 60424 entries, 0 to 60423
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   tipo_elem  60424 non-null  str    
 1   distrito   60364 non-null  float64
 2   id         60424 non-null  int64  
 3   cod_cent   60424 non-null  str    
 4   nombre     60157 non-null  str    
 5   utm_x      60424 non-null  float64
 6   utm_y      60424 non-null  float64
 7   longitud   60424 non-null  float64
 8   latitud    60424 non-null  float64
 9   archivo    60424 non-null  str    
dtypes: float64(5), int64(1), str(4)
memory usage: 9.0 MB


,tipo_elem,distrito,id,cod_cent,nombre,utm_x,utm_y,longitud,latitud,archivo
0,M30,5.0,6640,PM10013,PM10013,442865.853275,4.481386e+06,-3.674087,40.481200,ubicacion-abril-26.csv
1,M30,5.0,6641,PM10021,PM10021,442869.287910,4.481350e+06,-3.674043,40.480878,ubicacion-abril-26.csv
2,M30,5.0,6642,PM10091,PM10091,442835.412762,4.480813e+06,-3.674395,40.476038,ubicacion-abril-26.csv
3,M30,5.0,6643,PM10092,PM10092,442819.699444,4.480807e+06,-3.674579,40.475980,ubicacion-abril-26.csv
4,M30,5.0,6644,PM10141,PM10141,443054.603649,4.480238e+06,-3.671757,40.470874,ubicacion-abril-26.csv


In [6]:
conteo_meses = ubicaciones.groupby("id")["archivo"].nunique().value_counts().sort_index()
conteo_meses
#Analizar la frecuencia temporal de los sensores

archivo
1        7
2        2
3        6
4       25
5       24
6        1
7       10
8        7
9        6
10       2
11       7
12    4991
Name: count, dtype: int64

In [7]:
conteo_por_id = ubicaciones.groupby("id")["archivo"].nunique()

print(conteo_por_id.describe())

sensores_inestables = conteo_por_id[conteo_por_id < 12]
print("Sensores inestables:", len(sensores_inestables))
#Detectar sensores inestables o intermitentes

count    5088.000000
mean       11.875786
std         0.960954
min         1.000000
25%        12.000000
50%        12.000000
75%        12.000000
max        12.000000
Name: archivo, dtype: float64
Sensores inestables: 97


## 3. Validación y análisis de cambios

In [ ]:
coord_check = (
    ubicaciones.groupby("id")[["latitud", "longitud"]]
    .nunique()
)

cambios_coord = coord_check[
    (coord_check["latitud"] > 1) |
    (coord_check["longitud"] > 1)
]

print("Sensores con cambios de coordenadas:", len(cambios_coord))
#Verificar si existen variaciones en las coordenadas de los sensores entre los diferentes archivos

Sensores con cambios de coordenadas: 229


In [ ]:
movimiento = (
    ubicaciones.groupby("id")[["latitud", "longitud"]]
    .agg(["min", "max"])
)

movimiento["dif_lat"] = movimiento[("latitud","max")] - movimiento[("latitud","min")]
movimiento["dif_lon"] = movimiento[("longitud","max")] - movimiento[("longitud","min")]

movimiento[["dif_lat","dif_lon"]].sort_values(
    by="dif_lat", ascending=False
).head(10)
#Calcular la magnitud de las diferencias en latitud y longitud para identificar desplazamientos significativos

,dif_lat,dif_lon
,,
id,,
10954,0.000404,0.000145
11472,0.000394,0.000012
11452,0.000384,0.000020
11341,0.000353,0.000266
11455,0.000320,0.000433
4608,0.000288,0.000074
10867,0.000258,0.000175
4290,0.000198,0.000057


In [ ]:
nombre_var = ubicaciones.groupby("id")["nombre"].nunique()

print(nombre_var.describe())

sensores_nombre_inestable = nombre_var[nombre_var > 1]
print("Sensores con varios nombres:", len(sensores_nombre_inestable))
#Analizar la consistencia en los nombres asignados a cada sensor a lo largo del tiempo

count    5088.000000
mean        1.031053
std         0.188678
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         2.000000
Name: nombre, dtype: float64
Sensores con varios nombres: 172


In [ ]:
nombre_var = ubicaciones.groupby("id")["nombre"].nunique()

print(nombre_var.describe())

sensores_nombre_inestable = nombre_var[nombre_var > 1]
print("Sensores con varios nombres:", len(sensores_nombre_inestable))
#Analizar la consistencia en los nombres asignados a cada sensor a lo largo del tiempo

count    5088.000000
mean        1.031053
std         0.188678
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         2.000000
Name: nombre, dtype: float64
Sensores con varios nombres: 172


In [ ]:
id_test = sensores_nombre_inestable.index[0]

ubicaciones[ubicaciones["id"] == id_test][
    ["archivo","nombre","latitud","longitud"]
].sort_values("archivo")
#Inspeccionar un ejemplo específico de sensor con múltiples nombres para entender el comportamiento de los datos

,archivo,nombre,latitud,longitud
1711,ubicacion-abril-26.csv,"Santa Engracia, 26 S-N(Caracas-Pl. Chamberi)",40.431178,-3.697088
8064,ubicacion-agosto-25.csv,"Santa Engracia, 26 S-N - Caracas-Pl. Chamberi",40.431178,-3.697088
12633,ubicacion-diciembre-25.csv,"Santa Engracia, 26 S-N(Caracas-Pl. Chamberi)",40.431178,-3.697088
16211,ubicacion-enero-26.csv,"Santa Engracia, 26 S-N(Caracas-Pl. Chamberi)",40.431178,-3.697088
23882,ubicacion-febrero-26.csv,"Santa Engracia, 26 S-N(Caracas-Pl. Chamberi)",40.431178,-3.697088
25481,ubicacion-julio-25.csv,"Santa Engracia, 26 S-N - Caracas-Pl. Chamberi",40.431178,-3.697088
33978,ubicacion-junio-26.csv,"Santa Engracia, 26 S-N(Caracas-Pl. Chamberi)",40.431178,-3.697088
37303,ubicacion-marzo-26.csv,"Santa Engracia, 26 S-N(Caracas-Pl. Chamberi)",40.431178,-3.697088
41767,ubicacion-mayo-26.csv,"Santa Engracia, 26 S-N(Caracas-Pl. Chamberi)",40.431178,-3.697088
47605,ubicacion-noviembre-25.csv,"Santa Engracia, 26 S-N(Caracas-Pl. Chamberi)",40.431178,-3.697088


## 4. Normalización de datos

In [ ]:
def normalizar_texto(t):
    if pd.isna(t):
        return t
    t = t.lower()
    t = re.sub(r"\s+", " ", t)
    t = t.strip()
    return t

ubicaciones["nombre_norm"] = ubicaciones["nombre"].apply(normalizar_texto)
#Definir una función para estandarizar los textos de los nombres (minúsculas y eliminación de espacios múltiples).

## 5. Consolidación y creación de tabla maestra

In [ ]:
base_ids = ubicaciones[["id"]].drop_duplicates()
#Obtener el listado único de identificadores de sensores

In [ ]:
coord_final = (
    ubicaciones
    .groupby("id")[["latitud", "longitud"]]
    .median()
    .reset_index()
)
#Consolidar las coordenadas utilizando la mediana para mayor robustez ante valores atípicos

In [ ]:
nombre_final = (
    ubicaciones
    .groupby("id")["nombre_norm"]
    .agg(lambda x: x.dropna().mode().iloc[0] if not x.dropna().empty else None)
    .reset_index()
)
#Seleccionar el nombre más frecuente (moda) normalizado para cada sensor

In [ ]:
ubicacion_maestra = (
    ubicaciones
    .groupby("id")
    .agg({
        "tipo_elem": "first",
        "distrito": "first",
        "cod_cent": "first",
        "utm_x": "first",
        "utm_y": "first",
        "latitud": "median",
        "longitud": "median",
        "nombre_norm": lambda x: x.dropna().mode().iloc[0] if not x.dropna().empty else None
    })
    .reset_index()
)

print("Filas finales:", len(ubicacion_maestra))
print("IDs únicos:", ubicacion_maestra["id"].nunique())
#Construir la tabla maestra consolidando atributos fijos, coordenadas y nombres

Filas finales: 5088
IDs únicos: 5088


## 6. Verificación y exportación

In [ ]:
assert ubicacion_maestra["id"].is_unique
print("✔ Tabla maestra consistente")
#Validar que la tabla maestra tenga una clave primaria única por sensor

✔ Tabla maestra consistente


In [ ]:
ubicacion_maestra.to_csv(
    "../data/raw/ubicacion_maestra.csv",
    index=False,
    encoding="utf-8"
)
#Guardar la tabla maestra consolidada en formato CSV